<a href="https://colab.research.google.com/github/Mohamed-Shawky281/Hybrid-RAG-Research-assistant/blob/main/Hybrid_RAG_Research_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Hybrid RAG - Research Assistant Project

##Downloading libraries and dependencies...

In [1]:
!pip install -q langchain langchain-community langchain-chroma chromadb pypdf sentence-transformers rank_bm25
!pip install -q langchain-classic

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/rag_research_assistant"
os.makedirs(f"{PROJECT_DIR}/papers", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/processed", exist_ok=True)
print("Project folder ready at:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder ready at: /content/drive/MyDrive/rag_research_assistant


In [3]:
from pathlib import Path

pdf_paths = list(Path(f"{PROJECT_DIR}/papers").glob("*.pdf"))
print(f"{len(pdf_paths)} PDF(s) found in papers/:")
for p in pdf_paths:
    print(" -", p.name)

32 PDF(s) found in papers/:
 - CryptographyinPostQuantumComputingEra (1).pdf
 - DOC-20260827-WA0014_260915_154811 (1).pdf
 - DOC-20260827-WA0016_260915_155026 (1).pdf
 - DOC-20260822-WA0006_260915_154729 (1).pdf
 - DOC-20260827-WA0015_260915_155007 (1).pdf
 - CryptographyinPostQuantumComputingEra (2).pdf
 - DOC-20260822-WA0006_260915_154729 (2).pdf
 - DOC-20260827-WA0014_260915_154811 (2).pdf
 - DOC-20260827-WA0015_260915_155007 (2).pdf
 - DOC-20260827-WA0016_260915_155026 (2).pdf
 - CryptographyinPostQuantumComputingEra (3).pdf
 - DOC-20260822-WA0006_260915_154729 (3).pdf
 - DOC-20260827-WA0014_260915_154811 (3).pdf
 - DOC-20260827-WA0015_260915_155007 (3).pdf
 - DOC-20260827-WA0016_260915_155026 (3).pdf
 - CryptographyinPostQuantumComputingEra (4).pdf
 - DOC-20260822-WA0006_260915_154729 (4).pdf
 - DOC-20260827-WA0014_260915_154811 (4).pdf
 - DOC-20260827-WA0015_260915_155007 (4).pdf
 - DOC-20260827-WA0016_260915_155026 (4).pdf
 - CryptographyinPostQuantumComputingEra (5).pdf
 - DOC-

In [4]:
import hashlib
from pathlib import Path

pdf_paths = list(Path(f"{PROJECT_DIR}/papers").glob("*.pdf"))
seen_hashes = {}
duplicates = []

for path in pdf_paths:
    h = hashlib.md5(path.read_bytes()).hexdigest()
    if h in seen_hashes:
        duplicates.append(path)
    else:
        seen_hashes[h] = path

print(f"Found {len(pdf_paths)} files, {len(seen_hashes)} unique, {len(duplicates)} duplicates")
print("\nKeeping:")
for p in seen_hashes.values():
    print(" -", p.name)
print("\nWill delete:")
for d in duplicates:
    print(" -", d.name)

Found 32 files, 7 unique, 25 duplicates

Keeping:
 - CryptographyinPostQuantumComputingEra (1).pdf
 - DOC-20260827-WA0014_260915_154811 (1).pdf
 - DOC-20260827-WA0016_260915_155026 (1).pdf
 - DOC-20260822-WA0006_260915_154729 (1).pdf
 - DOC-20260827-WA0015_260915_155007 (1).pdf
 - 2025.acl-long.440.pdf
 - mental-2024-1-e57400.pdf

Will delete:
 - CryptographyinPostQuantumComputingEra (2).pdf
 - DOC-20260822-WA0006_260915_154729 (2).pdf
 - DOC-20260827-WA0014_260915_154811 (2).pdf
 - DOC-20260827-WA0015_260915_155007 (2).pdf
 - DOC-20260827-WA0016_260915_155026 (2).pdf
 - CryptographyinPostQuantumComputingEra (3).pdf
 - DOC-20260822-WA0006_260915_154729 (3).pdf
 - DOC-20260827-WA0014_260915_154811 (3).pdf
 - DOC-20260827-WA0015_260915_155007 (3).pdf
 - DOC-20260827-WA0016_260915_155026 (3).pdf
 - CryptographyinPostQuantumComputingEra (4).pdf
 - DOC-20260822-WA0006_260915_154729 (4).pdf
 - DOC-20260827-WA0014_260915_154811 (4).pdf
 - DOC-20260827-WA0015_260915_155007 (4).pdf
 - DOC-20260

In [5]:
for d in duplicates:
    d.unlink()
    print("Deleted:", d.name)

pdf_paths = list(Path(f"{PROJECT_DIR}/papers").glob("*.pdf"))
print(f"\n{len(pdf_paths)} PDF(s) remain in papers/")

Deleted: CryptographyinPostQuantumComputingEra (2).pdf
Deleted: DOC-20260822-WA0006_260915_154729 (2).pdf
Deleted: DOC-20260827-WA0014_260915_154811 (2).pdf
Deleted: DOC-20260827-WA0015_260915_155007 (2).pdf
Deleted: DOC-20260827-WA0016_260915_155026 (2).pdf
Deleted: CryptographyinPostQuantumComputingEra (3).pdf
Deleted: DOC-20260822-WA0006_260915_154729 (3).pdf
Deleted: DOC-20260827-WA0014_260915_154811 (3).pdf
Deleted: DOC-20260827-WA0015_260915_155007 (3).pdf
Deleted: DOC-20260827-WA0016_260915_155026 (3).pdf
Deleted: CryptographyinPostQuantumComputingEra (4).pdf
Deleted: DOC-20260822-WA0006_260915_154729 (4).pdf
Deleted: DOC-20260827-WA0014_260915_154811 (4).pdf
Deleted: DOC-20260827-WA0015_260915_155007 (4).pdf
Deleted: DOC-20260827-WA0016_260915_155026 (4).pdf
Deleted: CryptographyinPostQuantumComputingEra (5).pdf
Deleted: DOC-20260822-WA0006_260915_154729 (5).pdf
Deleted: DOC-20260827-WA0014_260915_154811 (5).pdf
Deleted: DOC-20260827-WA0015_260915_155007 (5).pdf
Deleted: DOC-20

Uploading The wanted reference documents

In [6]:
from google.colab import files

uploaded = files.upload()
for fname in uploaded.keys():
    dest = f"{PROJECT_DIR}/papers/{fname}"
    with open(dest, "wb") as f:
        f.write(uploaded[fname])
    print(f"Saved {fname} -> {dest}")

##Loading PDFs into LangChain Documents

In [7]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_paths = list(Path(f"{PROJECT_DIR}/papers").glob("*.pdf"))

all_docs = []
for path in pdf_paths:
    loader = PyPDFLoader(str(path))
    docs = loader.load()  # one Document per page, metadata already includes 'source' and 'page'
    all_docs.extend(docs)

print(f"Loaded {len(all_docs)} pages from {len(pdf_paths)} papers")
print(all_docs[0].page_content[:500])
print(all_docs[0].metadata)

/tmp/ipykernel_5543/455741346.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 172 pages from 7 papers
1 
CRYPTOGRAPHY IN POST-QUANTUM ERA 
 
 
 
 
 
Cryptography in Post Quantum Computing Era 
Neerav Sood 
Independent Researcher
{'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-01-24T13:31:21-05:00', 'author': 'TR', 'moddate': '2024-01-24T13:31:21-05:00', 'source': '/content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra (1).pdf', 'total_pages': 96, 'page': 0, 'page_label': '1'}


In [8]:
#Take into consideration like truncated output , so strip it to avoid retreivial of gaps or empty spaces
import re

def clean_text(text: str) -> str:
    """
    Cleans up common PDF-extraction artifacts before chunking:
    - collapses multiple blank lines/newlines into one
    - collapses runs of spaces/tabs into a single space
    - strips leading/trailing whitespace per line
    - joins words that got hyphen-broken across a line (common in PDFs)
    """
    # Fix hyphenated words split across a line break, e.g. "crypto-\ngraphy" -> "cryptography"
    text = re.sub(r"-\n", "", text)

    # Collapse 3+ newlines (big gaps) down to a single paragraph break
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Collapse runs of spaces/tabs into one space
    text = re.sub(r"[ \t]+", " ", text)

    # Strip trailing whitespace on each line
    text = "\n".join(line.strip() for line in text.split("\n"))

    return text.strip()


for doc in all_docs:
    doc.page_content = clean_text(doc.page_content)

print("Cleaned text for all", len(all_docs), "pages")
print(all_docs[0].page_content[:500])  # sanity check -- compare to before cleaning

Cleaned text for all 172 pages
1
CRYPTOGRAPHY IN POST-QUANTUM ERA





Cryptography in Post Quantum Computing Era
Neerav Sood
Independent Researcher


##Chunking (including overlapping) & tuned

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,      # characters -- LangChain's default unit, Also to be tuned
    chunk_overlap=500,
    separators=["\n\n", "\n", ". ", " ", ""],  # tries paragraph breaks first, falls back to smaller units
)

split_docs = splitter.split_documents(all_docs)
print(f"Created {len(split_docs)} chunks from {len(all_docs)} pages")
print(split_docs[0].page_content[:300])
print(split_docs[0].metadata)

Created 577 chunks from 172 pages
1
CRYPTOGRAPHY IN POST-QUANTUM ERA





Cryptography in Post Quantum Computing Era
Neerav Sood
Independent Researcher
{'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-01-24T13:31:21-05:00', 'author': 'TR', 'moddate': '2024-01-24T13:31:21-05:00', 'source': '/content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra (1).pdf', 'total_pages': 96, 'page': 0, 'page_label': '1'}


##Building the Chroma vector store (Meaning Similarity)

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    persist_directory=f"{PROJECT_DIR}/chroma_db",
)

print(f"Chroma vector store built with {vectorstore._collection.count()} chunks")

/tmp/ipykernel_5543/2980357329.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chroma vector store built with 19545 chunks


In [11]:
print(f"Chroma vector store built with {vectorstore._collection.count()} chunks")

Chroma vector store built with 19545 chunks


In [12]:
import pickle
import os

file_path = f"{PROJECT_DIR}/processed/split_docs.pkl"

# Check if the file exists. If not, and split_docs is available, save it.
# This assumes split_docs is defined in the current kernel state from previous cells.
if not os.path.exists(file_path):
    print(f"File '{file_path}' not found. Saving 'split_docs' to file first.")
    with open(file_path, "wb") as f:
        pickle.dump(split_docs, f)

# Now, attempt to load the file (which should now exist or existed already)
with open(file_path, "rb") as f:
    split_docs = pickle.load(f)

print(f"Loaded {len(split_docs)} chunks")

Loaded 1972 chunks


##Build the Keyword similarity (BM25 retriever)

In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = 5  # how many chunks BM25 returns per query

print("BM25 retriever built")

BM25 retriever built


In [14]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("Vector retriever built")

Vector retriever built


##Merging of Both methods and tuning

In [15]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.65, 0.35], # The ratio of relavence used between vector and bm-25
)

print("Ensemble retriever built")

Ensemble retriever built


In [16]:
#Testing of retriver and accurate retrivial on a paper
query = "What wrong with Traditional cryptographic systems compared to quantum ones"

results = ensemble_retriever.invoke(query)

print(f"Returned {len(results)} chunks\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')} | Page: {doc.metadata.get('page', '?')}")
    print(doc.page_content[:300])
    print()

Returned 5 chunks

--- Result 1 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra.pdf | Page: 19
20 
CRYPTOGRAPHY IN POST-QUANTUM ERA 
 
 
The robustness of lattice-based cryptographic systems against both classical and quantum 
attacks, coupled with their adaptability to a wide range of cryptographic applications, makes 
them a cornerstone in the quest for secure communication and data protect

--- Result 2 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra.pdf | Page: 19
The robustness of lattice-based cryptographic systems against both classical and quantum
attacks, coupled with their adaptability to a wide range of cryptographic applications, makes
them a cornerstone in the quest for secure communication and data protection in the quantum
age. Their theoretical fo

--- Result 3 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra

In [17]:
#Testing of retriver and accurate retrivial on a paper
query = "What problems are faced in multi-resource LLMs? "

results = ensemble_retriever.invoke(query)

print(f"Returned {len(results)} chunks\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')} | Page: {doc.metadata.get('page', '?')}")
    print(doc.page_content[:300])
    print()

Returned 4 chunks

--- Result 1 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/DOC-20260827-WA0014_260915_154811 (1).pdf | Page: 3
necessitating extensive adjustments to other components.
1) Planning: First of all, to incorporate the cases which
does not require any sources of external knowledge, we define
several additional indicate tokens, corresponding to different
sources, including the NULL token which signifies that there

--- Result 2 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra.pdf | Page: 14
complexity, making these problems intractable for both classical and quantum computers. 
The reason why lattice-based cryptographic methods are resistant to quantum computing 
attacks lies in the nature of these lattice problems. Unlike problems such as integer factorization 
or discrete logarithms,

--- Result 3 ---
Source: /content/drive/MyDrive/rag_research_assistant/papers/CryptographyinPostQuantumComputingEra.

##Setting API env for Generation

In [18]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.7 MB/s eta 0:00:00


In [19]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

print("Groq client ready")

Groq client ready


In [20]:
import requests

resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {api_key}"}
)
models = resp.json()

for m in models["data"]:
    print(m["id"])

openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-safeguard-20b
groq/compound
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m
groq/compound-mini
canopylabs/orpheus-arabic-saudi
qwen/qwen3.8-27b
allam-2-7b
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
whisper-large-v3


In [21]:
#Testing of API
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]
)
print(response.choices[0].message.content)

Hello!


##Prompt Engineering

In [22]:
def build_context(docs):
    """Formats retrieved chunks into a numbered context block with citations."""
    context_parts = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "unknown").split("/")[-1]
        page = doc.metadata.get("page_label", doc.metadata.get("page", "?"))
        context_parts.append(f"[{i+1}] (Source: {source}, Page: {page})\n{doc.page_content}")
    return "\n\n".join(context_parts)


def build_prompt(query, docs):
    context = build_context(docs)
    prompt = f"""You are a research assistant answering questions based ONLY on the provided paper excerpts below.

Rules:
- Answer using ONLY information found in the excerpts below.
- If the excerpts don't contain enough information to answer, say so clearly -- do not use outside knowledge.
- Cite your sources using the [number] format matching the excerpts (e.g. "Quantum computing threatens RSA encryption [1].").
- Give a thorough, well-explained answer -- don't just state a conclusion in one line. Explain the reasoning and relevant details found in the excerpts, and elaborate on how the evidence supports your answer.

Excerpts:
{context}

Question: {query}

Answer:"""
    return prompt

In [23]:
def ask(query, retriever=ensemble_retriever, llm_model="openai/gpt-oss-120b"):
    docs = retriever.invoke(query)
    prompt = build_prompt(query, docs)

    response = client.chat.completions.create(
        model=llm_model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1024,
    )

    answer = response.choices[0].message.content
    return answer, docs

##Simple Friendly UI

In [24]:
!pip install -q gradio

In [25]:
import gradio as gr
import os
import shutil
import pickle

# ---------- Styling ----------
custom_css = """
#main-title { font-size: 2.2em; font-weight: 700; color: #1a1a2e; }
.gradio-container { font-family: 'Inter', sans-serif; background: #f7f7fb; }
#answer-box textarea { font-size: 1.05em; line-height: 1.6; }
"""

# ---------- Upload + Index handler ----------
def process_uploaded_pdfs(files):
    if not files:
        return "No files selected."

    global split_docs, bm25_retriever, ensemble_retriever

    all_new_split_docs = []
    processed_names = []

    for file in files:
        pdf_path = file.name
        filename = os.path.basename(pdf_path)

        dest = f"{PROJECT_DIR}/papers/{filename}"
        shutil.copy(pdf_path, dest)

        loader = PyPDFLoader(dest)
        new_docs = loader.load()
        for doc in new_docs:
            doc.page_content = clean_text(doc.page_content)

        new_split_docs = splitter.split_documents(new_docs)
        all_new_split_docs.extend(new_split_docs)
        processed_names.append(f"{filename} ({len(new_split_docs)} chunks)")

    # One batched embedding call for all new chunks, not one per file
    vectorstore.add_documents(all_new_split_docs)

    split_docs = split_docs + all_new_split_docs

    with open(f"{PROJECT_DIR}/processed/split_docs.pkl", "wb") as f:
        pickle.dump(split_docs, f)

    bm25_retriever = BM25Retriever.from_documents(split_docs)
    bm25_retriever.k = 5

    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[0.65, 0.35],
    )

    summary = "\n".join(processed_names)
    return f"✅ Processed {len(files)} file(s):\n{summary}\n\nTotal chunks in system: {len(split_docs)}."


# ---------- Ask handler ----------
def rag_interface(query):
    if not query.strip():
        return "Please enter a question.", ""

    answer, docs = ask(query)

    sources_text = ""
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "unknown").split("/")[-1]
        page = doc.metadata.get("page_label", doc.metadata.get("page", "?"))
        sources_text += f"**[{i+1}]** {source}, page {page}\n\n"

    return answer, sources_text


# ---------- UI layout ----------
with gr.Blocks(css=custom_css, theme=gr.themes.Soft(), title="Research Assistant") as demo:
    gr.Markdown(
        """
        # 📚 Hybrid RAG Research Assistant
        Upload research papers and ask grounded questions about them, with citations.
        """,
        elem_id="main-title",
    )

    with gr.Tabs():
        with gr.Tab("📤 Upload Papers"):
            gr.Markdown("Upload one or more PDFs to add them to the searchable knowledge base.")
            file_input = gr.File(label="Upload PDF(s)", file_types=[".pdf"], file_count="multiple")
            upload_btn = gr.Button("Process & Index", variant="primary")
            upload_status = gr.Textbox(label="Status", lines=4)

            upload_btn.click(fn=process_uploaded_pdfs, inputs=file_input, outputs=upload_status)

        with gr.Tab("💬 Ask a Question"):
            with gr.Row():
                with gr.Column(scale=2):
                    query_input = gr.Textbox(
                        label="Your question",
                        placeholder="e.g. What post-quantum algorithms are discussed?",
                        lines=2,
                    )
                    submit_btn = gr.Button("Ask", variant="primary")

                    gr.Examples(
                        examples=[
                            "What post-quantum cryptographic algorithms are discussed?",
                            "What is the main contribution of these papers?",
                            "What evaluation methods were used?",
                        ],
                        inputs=query_input,
                    )

                with gr.Column(scale=3):
                    answer_output = gr.Textbox(label="Answer", lines=10, elem_id="answer-box")
                    sources_output = gr.Markdown(label="Sources")

            submit_btn.click(fn=rag_interface, inputs=query_input, outputs=[answer_output, sources_output])
            query_input.submit(fn=rag_interface, inputs=query_input, outputs=[answer_output, sources_output])

demo.launch(share=True)

/tmp/ipykernel_5543/4161389990.py:76: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft(), title="Research Assistant") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6d6c7e172516c292b7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
